In [ ]:
import time, math, os
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.metrics import (
    accuracy_score, roc_auc_score, average_precision_score,
    confusion_matrix, classification_report, roc_curve, precision_recall_curve
)

In [ ]:
def pooled_dim(pooling, T, D):
    return D if pooling in ("mean","max","cls","gem") else (2*D if pooling in ("mean_max","mean_std") else None)

def pool_batch(Xb, pooling="mean"):
    if Xb.ndim == 2: return Xb
    if pooling == "mean":      return Xb.mean(axis=1)
    if pooling == "max":       return Xb.max(axis=1)
    if pooling == "cls":       return Xb[:, 0, :]
    if pooling == "mean_max":  return np.concatenate([Xb.mean(axis=1), Xb.max(axis=1)], axis=1)
    if pooling == "mean_std":  return np.concatenate([Xb.mean(axis=1), Xb.std(axis=1)], axis=1)
    if pooling == "gem":
        p, eps = 3.0, 1e-6
        X = np.clip(Xb, a_min=0.0, a_max=None)
        return np.power((np.power(X + eps, p).mean(axis=1) + eps), 1.0 / p)
    raise ValueError(f"Unknown pooling: {pooling}")

def choose_batch_size(X_mm, target_bytes=512 * 1024**2):
    if X_mm.ndim == 3:
        _, T, D = X_mm.shape
        bytes_per_sample = T * D * X_mm.dtype.itemsize
    else:
        bytes_per_sample = X_mm.shape[1] * X_mm.dtype.itemsize
    b = max(1, int(target_bytes // max(1, bytes_per_sample)))
    for cand in (256,128,64,32,16,8,4,2,1):
        if b >= cand: return cand
    return 1

def pool_memmap_streaming_timed(X_mm, pooling="mean", batch=None, out_dtype=np.float32, desc="Pooling"):
    N = X_mm.shape[0]
    if X_mm.ndim == 3:
        T, D = X_mm.shape[1], X_mm.shape[2]
        out_dim = pooled_dim(pooling, T, D)
    elif X_mm.ndim == 2:
        out_dim = X_mm.shape[1]
    else:
        raise ValueError(f"Unexpected shape: {X_mm.shape}")

    if batch is None:
        batch = choose_batch_size(X_mm, target_bytes=512 * 1024**2)

    out = np.empty((N, out_dim), dtype=out_dtype)
    start = time.perf_counter()
    with tqdm(total=N, desc=f"{desc} ({pooling})", unit="rows") as pbar:
        for s in range(0, N, batch):
            e = min(s + batch, N)
            pooled = pool_batch(X_mm[s:e], pooling=pooling)
            pooled = np.nan_to_num(pooled, nan=0.0, posinf=1e6, neginf=-1e6)
            out[s:e] = pooled.astype(out_dtype, copy=False)
            pbar.update(e - s)
    elapsed = time.perf_counter() - start
    rows_per_sec = N / max(elapsed, 1e-9)
    mb = out.nbytes / (1024**2)
    print(f"{desc} done in {elapsed:.2f}s  (~{rows_per_sec:.1f} rows/s, output ~{mb:.1f} MB)")
    return out, elapsed

### Random circle r = 1

In [ ]:
TR_EMB = "/home/jupyter-yin10/Image_Analysis/random_c1/train_emb_rad_dino.npy"
TR_LAB = "/home/jupyter-yin10/Image_Analysis/random_c1/train_labels_rad_dino.npy"
TE_EMB = "/home/jupyter-yin10/Image_Analysis/random_c1/test_emb_rad_dino.npy"
TE_LAB = "/home/jupyter-yin10/Image_Analysis/random_c1/test_labels_rad_dino.npy"

def load_memmap(path):
    return np.load(path, mmap_mode="r", allow_pickle=False)

Xtr_mm = load_memmap(TR_EMB)
Xte_mm = load_memmap(TE_EMB)
y_tr   = np.load(TR_LAB, allow_pickle=False)
y_te   = np.load(TE_LAB, allow_pickle=False)

# ---------- run + time ----------
POOLING = "mean"     # try: "mean_std" for stronger baseline (bigger features)
BATCH   = None       # None => auto (~512MB/batch). Use 64/32 if memory is tight.

X_tr, t_pool_tr = pool_memmap_streaming_timed(Xtr_mm, pooling=POOLING, batch=BATCH, out_dtype=np.float32, desc="Pool train")
X_te, t_pool_te = pool_memmap_streaming_timed(Xte_mm, pooling=POOLING, batch=BATCH, out_dtype=np.float32, desc="Pool test")
print(f"Pooled shapes: train {X_tr.shape}, test {X_te.shape}")

# ---- train & time LogisticRegression (CPU) ----
t0 = time.perf_counter()
probe = make_pipeline(
    StandardScaler(with_mean=True, with_std=True),
    LogisticRegression(
        penalty="l2",
        C=1.0,
        solver="lbfgs",         # try 'saga' if want L1/elasticnet or faster on very large N
        max_iter=2000,
        class_weight="balanced",
        random_state=0,
        n_jobs=-1
    )
)
probe.fit(X_tr, y_tr)
t_fit = time.perf_counter() - t0
print(f"Training done in {t_fit:.2f}s")

# ---- eval & time ----
t0 = time.perf_counter()
proba = probe.predict_proba(X_te)[:, 1]
pred  = (proba >= 0.5).astype(int)
t_pred = time.perf_counter() - t0

acc   = accuracy_score(y_te, pred)
auroc = roc_auc_score(y_te, proba)
auprc = average_precision_score(y_te, proba)
cm    = confusion_matrix(y_te, pred)
print(f"Eval done in {t_pred:.2f}s")
print(f"Accuracy: {acc:.4f} | AUROC: {auroc:.4f} | AUPRC: {auprc:.4f}\nConfusion matrix:\n{cm}")
print(classification_report(y_te, pred, digits=3))

# ---- plots (unchanged) ----
fpr, tpr, _ = roc_curve(y_te, proba)
plt.figure(figsize=(5, 4))
plt.plot(fpr, tpr, label=f"AUC = {auroc:.3f}")
plt.plot([0, 1], [0, 1], linestyle="--", label="Chance")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title(f"ROC — Mole Present vs Absent ({POOLING}) [RADDINO-c1]")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

prec, rec, _ = precision_recall_curve(y_te, proba)
plt.figure(figsize=(5, 4))
plt.plot(rec, prec, label=f"AUPRC = {auprc:.3f}")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title(f"Precision–Recall — Mole Present vs Absent ({POOLING}) [RADDINO-c1]")
plt.legend(loc="lower left")
plt.tight_layout()
plt.show()

total_time = t_pool_tr + t_pool_te + t_fit + t_pred
print(f"\nTotal wall time ≈ {total_time:.2f}s")


### Random circle r = 2

In [ ]:
TR_EMB = "/home/jupyter-yin10/Image_Analysis/random_c2/train_emb_rad_dino.npy"
TR_LAB = "/home/jupyter-yin10/Image_Analysis/random_c2/train_labels_rad_dino.npy"
TE_EMB = "/home/jupyter-yin10/Image_Analysis/random_c2/test_emb_rad_dino.npy"
TE_LAB = "/home/jupyter-yin10/Image_Analysis/random_c2/test_labels_rad_dino.npy"

def load_memmap(path):
    return np.load(path, mmap_mode="r", allow_pickle=False)

Xtr_mm = load_memmap(TR_EMB)
Xte_mm = load_memmap(TE_EMB)
y_tr   = np.load(TR_LAB, allow_pickle=False)
y_te   = np.load(TE_LAB, allow_pickle=False)

# ---------- run + time ----------
POOLING = "mean"     # try: "mean_std" for stronger baseline (bigger features)
BATCH   = None       # None => auto (~512MB/batch). Use 64/32 if memory is tight.

X_tr, t_pool_tr = pool_memmap_streaming_timed(Xtr_mm, pooling=POOLING, batch=BATCH, out_dtype=np.float32, desc="Pool train")
X_te, t_pool_te = pool_memmap_streaming_timed(Xte_mm, pooling=POOLING, batch=BATCH, out_dtype=np.float32, desc="Pool test")
print(f"Pooled shapes: train {X_tr.shape}, test {X_te.shape}")

# ---- train & time LogisticRegression (CPU) ----
t0 = time.perf_counter()
probe = make_pipeline(
    StandardScaler(with_mean=True, with_std=True),
    LogisticRegression(
        penalty="l2",
        C=1.0,
        solver="lbfgs",         # try 'saga' if want L1/elasticnet or faster on very large N
        max_iter=2000,
        class_weight="balanced",
        random_state=0,
        n_jobs=-1
    )
)
probe.fit(X_tr, y_tr)
t_fit = time.perf_counter() - t0
print(f"Training done in {t_fit:.2f}s")

# ---- eval & time ----
t0 = time.perf_counter()
proba = probe.predict_proba(X_te)[:, 1]
pred  = (proba >= 0.5).astype(int)
t_pred = time.perf_counter() - t0

acc   = accuracy_score(y_te, pred)
auroc = roc_auc_score(y_te, proba)
auprc = average_precision_score(y_te, proba)
cm    = confusion_matrix(y_te, pred)
print(f"Eval done in {t_pred:.2f}s")
print(f"Accuracy: {acc:.4f} | AUROC: {auroc:.4f} | AUPRC: {auprc:.4f}\nConfusion matrix:\n{cm}")
print(classification_report(y_te, pred, digits=3))

# ---- plots (unchanged) ----
fpr, tpr, _ = roc_curve(y_te, proba)
plt.figure(figsize=(5, 4))
plt.plot(fpr, tpr, label=f"AUC = {auroc:.3f}")
plt.plot([0, 1], [0, 1], linestyle="--", label="Chance")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title(f"ROC — Mole Present vs Absent ({POOLING}) [RADDINO-c2]")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

prec, rec, _ = precision_recall_curve(y_te, proba)
plt.figure(figsize=(5, 4))
plt.plot(rec, prec, label=f"AUPRC = {auprc:.3f}")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title(f"Precision–Recall — Mole Present vs Absent ({POOLING}) [RADDINO-c2]")
plt.legend(loc="lower left")
plt.tight_layout()
plt.show()

total_time = t_pool_tr + t_pool_te + t_fit + t_pred
print(f"\nTotal wall time ≈ {total_time:.2f}s")


### Random pentagon R = 1

In [ ]:
TR_EMB = "/home/jupyter-yin10/Image_Analysis/random_p1/train_emb_rad_dino.npy"
TR_LAB = "/home/jupyter-yin10/Image_Analysis/random_p1/train_labels_rad_dino.npy"
TE_EMB = "/home/jupyter-yin10/Image_Analysis/random_p1/test_emb_rad_dino.npy"
TE_LAB = "/home/jupyter-yin10/Image_Analysis/random_p1/test_labels_rad_dino.npy"

def load_memmap(path):
    return np.load(path, mmap_mode="r", allow_pickle=False)

Xtr_mm = load_memmap(TR_EMB)
Xte_mm = load_memmap(TE_EMB)
y_tr   = np.load(TR_LAB, allow_pickle=False)
y_te   = np.load(TE_LAB, allow_pickle=False)

# ---------- run + time ----------
POOLING = "mean"     # try: "mean_std" for stronger baseline (bigger features)
BATCH   = None       # None => auto (~512MB/batch). Use 64/32 if memory is tight.

X_tr, t_pool_tr = pool_memmap_streaming_timed(Xtr_mm, pooling=POOLING, batch=BATCH, out_dtype=np.float32, desc="Pool train")
X_te, t_pool_te = pool_memmap_streaming_timed(Xte_mm, pooling=POOLING, batch=BATCH, out_dtype=np.float32, desc="Pool test")
print(f"Pooled shapes: train {X_tr.shape}, test {X_te.shape}")

# ---- train & time LogisticRegression (CPU) ----
t0 = time.perf_counter()
probe = make_pipeline(
    StandardScaler(with_mean=True, with_std=True),
    LogisticRegression(
        penalty="l2",
        C=1.0,
        solver="lbfgs",         # try 'saga' if want L1/elasticnet or faster on very large N
        max_iter=2000,
        class_weight="balanced",
        random_state=0,
        n_jobs=-1
    )
)
probe.fit(X_tr, y_tr)
t_fit = time.perf_counter() - t0
print(f"Training done in {t_fit:.2f}s")

# ---- eval & time ----
t0 = time.perf_counter()
proba = probe.predict_proba(X_te)[:, 1]
pred  = (proba >= 0.5).astype(int)
t_pred = time.perf_counter() - t0

acc   = accuracy_score(y_te, pred)
auroc = roc_auc_score(y_te, proba)
auprc = average_precision_score(y_te, proba)
cm    = confusion_matrix(y_te, pred)
print(f"Eval done in {t_pred:.2f}s")
print(f"Accuracy: {acc:.4f} | AUROC: {auroc:.4f} | AUPRC: {auprc:.4f}\nConfusion matrix:\n{cm}")
print(classification_report(y_te, pred, digits=3))

# ---- plots (unchanged) ----
fpr, tpr, _ = roc_curve(y_te, proba)
plt.figure(figsize=(5, 4))
plt.plot(fpr, tpr, label=f"AUC = {auroc:.3f}")
plt.plot([0, 1], [0, 1], linestyle="--", label="Chance")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title(f"ROC — Mole Present vs Absent ({POOLING}) [RADDINO-p1]")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

prec, rec, _ = precision_recall_curve(y_te, proba)
plt.figure(figsize=(5, 4))
plt.plot(rec, prec, label=f"AUPRC = {auprc:.3f}")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title(f"Precision–Recall — Mole Present vs Absent ({POOLING}) [RADDINO-p1]")
plt.legend(loc="lower left")
plt.tight_layout()
plt.show()

total_time = t_pool_tr + t_pool_te + t_fit + t_pred
print(f"\nTotal wall time ≈ {total_time:.2f}s")


### Random pentagon R = 2

In [ ]:
TR_EMB = "/home/jupyter-yin10/Image_Analysis/random_p2/train_emb_rad_dino.npy"
TR_LAB = "/home/jupyter-yin10/Image_Analysis/random_p2/train_labels_rad_dino.npy"
TE_EMB = "/home/jupyter-yin10/Image_Analysis/random_p2/test_emb_rad_dino.npy"
TE_LAB = "/home/jupyter-yin10/Image_Analysis/random_p2/test_labels_rad_dino.npy"

def load_memmap(path):
    return np.load(path, mmap_mode="r", allow_pickle=False)

Xtr_mm = load_memmap(TR_EMB)
Xte_mm = load_memmap(TE_EMB)
y_tr   = np.load(TR_LAB, allow_pickle=False)
y_te   = np.load(TE_LAB, allow_pickle=False)

# ---------- run + time ----------
POOLING = "mean"     # try: "mean_std" for stronger baseline (bigger features)
BATCH   = None       # None => auto (~512MB/batch). Use 64/32 if memory is tight.

X_tr, t_pool_tr = pool_memmap_streaming_timed(Xtr_mm, pooling=POOLING, batch=BATCH, out_dtype=np.float32, desc="Pool train")
X_te, t_pool_te = pool_memmap_streaming_timed(Xte_mm, pooling=POOLING, batch=BATCH, out_dtype=np.float32, desc="Pool test")
print(f"Pooled shapes: train {X_tr.shape}, test {X_te.shape}")

# ---- train & time LogisticRegression (CPU) ----
t0 = time.perf_counter()
probe = make_pipeline(
    StandardScaler(with_mean=True, with_std=True),
    LogisticRegression(
        penalty="l2",
        C=1.0,
        solver="lbfgs",         # try 'saga' if want L1/elasticnet or faster on very large N
        max_iter=2000,
        class_weight="balanced",
        random_state=0,
        n_jobs=-1
    )
)
probe.fit(X_tr, y_tr)
t_fit = time.perf_counter() - t0
print(f"Training done in {t_fit:.2f}s")

# ---- eval & time ----
t0 = time.perf_counter()
proba = probe.predict_proba(X_te)[:, 1]
pred  = (proba >= 0.5).astype(int)
t_pred = time.perf_counter() - t0

acc   = accuracy_score(y_te, pred)
auroc = roc_auc_score(y_te, proba)
auprc = average_precision_score(y_te, proba)
cm    = confusion_matrix(y_te, pred)
print(f"Eval done in {t_pred:.2f}s")
print(f"Accuracy: {acc:.4f} | AUROC: {auroc:.4f} | AUPRC: {auprc:.4f}\nConfusion matrix:\n{cm}")
print(classification_report(y_te, pred, digits=3))

# ---- plots (unchanged) ----
fpr, tpr, _ = roc_curve(y_te, proba)
plt.figure(figsize=(5, 4))
plt.plot(fpr, tpr, label=f"AUC = {auroc:.3f}")
plt.plot([0, 1], [0, 1], linestyle="--", label="Chance")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title(f"ROC — Mole Present vs Absent ({POOLING}) [RADDINO-p2]")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

prec, rec, _ = precision_recall_curve(y_te, proba)
plt.figure(figsize=(5, 4))
plt.plot(rec, prec, label=f"AUPRC = {auprc:.3f}")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title(f"Precision–Recall — Mole Present vs Absent ({POOLING}) [RADDINO-p2]")
plt.legend(loc="lower left")
plt.tight_layout()
plt.show()

total_time = t_pool_tr + t_pool_te + t_fit + t_pred
print(f"\nTotal wall time ≈ {total_time:.2f}s")


### Random Square 2*2

In [ ]:
TR_EMB = "/home/jupyter-yin10/Image_Analysis/random_s2/train_emb_rad_dino.npy"
TR_LAB = "/home/jupyter-yin10/Image_Analysis/random_s2/train_labels_rad_dino.npy"
TE_EMB = "/home/jupyter-yin10/Image_Analysis/random_s2/test_emb_rad_dino.npy"
TE_LAB = "/home/jupyter-yin10/Image_Analysis/random_s2/test_labels_rad_dino.npy"

def load_memmap(path):
    return np.load(path, mmap_mode="r", allow_pickle=False)

Xtr_mm = load_memmap(TR_EMB)
Xte_mm = load_memmap(TE_EMB)
y_tr   = np.load(TR_LAB, allow_pickle=False)
y_te   = np.load(TE_LAB, allow_pickle=False)

# ---------- run + time ----------
POOLING = "mean"     # try: "mean_std" for stronger baseline (bigger features)
BATCH   = None       # None => auto (~512MB/batch). Use 64/32 if memory is tight.

X_tr, t_pool_tr = pool_memmap_streaming_timed(Xtr_mm, pooling=POOLING, batch=BATCH, out_dtype=np.float32, desc="Pool train")
X_te, t_pool_te = pool_memmap_streaming_timed(Xte_mm, pooling=POOLING, batch=BATCH, out_dtype=np.float32, desc="Pool test")
print(f"Pooled shapes: train {X_tr.shape}, test {X_te.shape}")

# ---- train & time LogisticRegression (CPU) ----
t0 = time.perf_counter()
probe = make_pipeline(
    StandardScaler(with_mean=True, with_std=True),
    LogisticRegression(
        penalty="l2",
        C=1.0,
        solver="lbfgs",         # try 'saga' if want L1/elasticnet or faster on very large N
        max_iter=2000,
        class_weight="balanced",
        random_state=0,
        n_jobs=-1
    )
)
probe.fit(X_tr, y_tr)
t_fit = time.perf_counter() - t0
print(f"Training done in {t_fit:.2f}s")

# ---- eval & time ----
t0 = time.perf_counter()
proba = probe.predict_proba(X_te)[:, 1]
pred  = (proba >= 0.5).astype(int)
t_pred = time.perf_counter() - t0

acc   = accuracy_score(y_te, pred)
auroc = roc_auc_score(y_te, proba)
auprc = average_precision_score(y_te, proba)
cm    = confusion_matrix(y_te, pred)
print(f"Eval done in {t_pred:.2f}s")
print(f"Accuracy: {acc:.4f} | AUROC: {auroc:.4f} | AUPRC: {auprc:.4f}\nConfusion matrix:\n{cm}")
print(classification_report(y_te, pred, digits=3))

# ---- plots (unchanged) ----
fpr, tpr, _ = roc_curve(y_te, proba)
plt.figure(figsize=(5, 4))
plt.plot(fpr, tpr, label=f"AUC = {auroc:.3f}")
plt.plot([0, 1], [0, 1], linestyle="--", label="Chance")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title(f"ROC — Mole Present vs Absent ({POOLING}) [RADDINO-s2]")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

prec, rec, _ = precision_recall_curve(y_te, proba)
plt.figure(figsize=(5, 4))
plt.plot(rec, prec, label=f"AUPRC = {auprc:.3f}")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title(f"Precision–Recall — Mole Present vs Absent ({POOLING}) [RADDINO-s2]")
plt.legend(loc="lower left")
plt.tight_layout()
plt.show()

total_time = t_pool_tr + t_pool_te + t_fit + t_pred
print(f"\nTotal wall time ≈ {total_time:.2f}s")


### Random square 4*4

In [ ]:
TR_EMB = "/home/jupyter-yin10/Image_Analysis/random_s4/train_emb_rad_dino.npy"
TR_LAB = "/home/jupyter-yin10/Image_Analysis/random_s4/train_labels_rad_dino.npy"
TE_EMB = "/home/jupyter-yin10/Image_Analysis/random_s4/test_emb_rad_dino.npy"
TE_LAB = "/home/jupyter-yin10/Image_Analysis/random_s4/test_labels_rad_dino.npy"

def load_memmap(path):
    return np.load(path, mmap_mode="r", allow_pickle=False)

Xtr_mm = load_memmap(TR_EMB)
Xte_mm = load_memmap(TE_EMB)
y_tr   = np.load(TR_LAB, allow_pickle=False)
y_te   = np.load(TE_LAB, allow_pickle=False)

# ---------- run + time ----------
POOLING = "mean"     # try: "mean_std" for stronger baseline (bigger features)
BATCH   = None       # None => auto (~512MB/batch). Use 64/32 if memory is tight.

X_tr, t_pool_tr = pool_memmap_streaming_timed(Xtr_mm, pooling=POOLING, batch=BATCH, out_dtype=np.float32, desc="Pool train")
X_te, t_pool_te = pool_memmap_streaming_timed(Xte_mm, pooling=POOLING, batch=BATCH, out_dtype=np.float32, desc="Pool test")
print(f"Pooled shapes: train {X_tr.shape}, test {X_te.shape}")

# ---- train & time LogisticRegression (CPU) ----
t0 = time.perf_counter()
probe = make_pipeline(
    StandardScaler(with_mean=True, with_std=True),
    LogisticRegression(
        penalty="l2",
        C=1.0,
        solver="lbfgs",         # try 'saga' if want L1/elasticnet or faster on very large N
        max_iter=2000,
        class_weight="balanced",
        random_state=0,
        n_jobs=-1
    )
)
probe.fit(X_tr, y_tr)
t_fit = time.perf_counter() - t0
print(f"Training done in {t_fit:.2f}s")

# ---- eval & time ----
t0 = time.perf_counter()
proba = probe.predict_proba(X_te)[:, 1]
pred  = (proba >= 0.5).astype(int)
t_pred = time.perf_counter() - t0

acc   = accuracy_score(y_te, pred)
auroc = roc_auc_score(y_te, proba)
auprc = average_precision_score(y_te, proba)
cm    = confusion_matrix(y_te, pred)
print(f"Eval done in {t_pred:.2f}s")
print(f"Accuracy: {acc:.4f} | AUROC: {auroc:.4f} | AUPRC: {auprc:.4f}\nConfusion matrix:\n{cm}")
print(classification_report(y_te, pred, digits=3))

# ---- plots (unchanged) ----
fpr, tpr, _ = roc_curve(y_te, proba)
plt.figure(figsize=(5, 4))
plt.plot(fpr, tpr, label=f"AUC = {auroc:.3f}")
plt.plot([0, 1], [0, 1], linestyle="--", label="Chance")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title(f"ROC — Mole Present vs Absent ({POOLING}) [RADDINO-s4]")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

prec, rec, _ = precision_recall_curve(y_te, proba)
plt.figure(figsize=(5, 4))
plt.plot(rec, prec, label=f"AUPRC = {auprc:.3f}")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title(f"Precision–Recall — Mole Present vs Absent ({POOLING}) [RADDINO-s4]")
plt.legend(loc="lower left")
plt.tight_layout()
plt.show()

total_time = t_pool_tr + t_pool_te + t_fit + t_pred
print(f"\nTotal wall time ≈ {total_time:.2f}s")


### Random Square 8*8

In [ ]:
TR_EMB = "/home/jupyter-yin10/Image_Analysis/random_s8/train_emb_rad_dino.npy"
TR_LAB = "/home/jupyter-yin10/Image_Analysis/random_s8/train_labels_rad_dino.npy"
TE_EMB = "/home/jupyter-yin10/Image_Analysis/random_s8/test_emb_rad_dino.npy"
TE_LAB = "/home/jupyter-yin10/Image_Analysis/random_s8/test_labels_rad_dino.npy"

def load_memmap(path):
    return np.load(path, mmap_mode="r", allow_pickle=False)

Xtr_mm = load_memmap(TR_EMB)
Xte_mm = load_memmap(TE_EMB)
y_tr   = np.load(TR_LAB, allow_pickle=False)
y_te   = np.load(TE_LAB, allow_pickle=False)

# ---------- run + time ----------
POOLING = "mean"     # try: "mean_std" for stronger baseline (bigger features)
BATCH   = None       # None => auto (~512MB/batch). Use 64/32 if memory is tight.

X_tr, t_pool_tr = pool_memmap_streaming_timed(Xtr_mm, pooling=POOLING, batch=BATCH, out_dtype=np.float32, desc="Pool train")
X_te, t_pool_te = pool_memmap_streaming_timed(Xte_mm, pooling=POOLING, batch=BATCH, out_dtype=np.float32, desc="Pool test")
print(f"Pooled shapes: train {X_tr.shape}, test {X_te.shape}")

# ---- train & time LogisticRegression (CPU) ----
t0 = time.perf_counter()
probe = make_pipeline(
    StandardScaler(with_mean=True, with_std=True),
    LogisticRegression(
        penalty="l2",
        C=1.0,
        solver="lbfgs",         # try 'saga' if want L1/elasticnet or faster on very large N
        max_iter=2000,
        class_weight="balanced",
        random_state=0,
        n_jobs=-1
    )
)
probe.fit(X_tr, y_tr)
t_fit = time.perf_counter() - t0
print(f"Training done in {t_fit:.2f}s")

# ---- eval & time ----
t0 = time.perf_counter()
proba = probe.predict_proba(X_te)[:, 1]
pred  = (proba >= 0.5).astype(int)
t_pred = time.perf_counter() - t0

acc   = accuracy_score(y_te, pred)
auroc = roc_auc_score(y_te, proba)
auprc = average_precision_score(y_te, proba)
cm    = confusion_matrix(y_te, pred)
print(f"Eval done in {t_pred:.2f}s")
print(f"Accuracy: {acc:.4f} | AUROC: {auroc:.4f} | AUPRC: {auprc:.4f}\nConfusion matrix:\n{cm}")
print(classification_report(y_te, pred, digits=3))

# ---- plots (unchanged) ----
fpr, tpr, _ = roc_curve(y_te, proba)
plt.figure(figsize=(5, 4))
plt.plot(fpr, tpr, label=f"AUC = {auroc:.3f}")
plt.plot([0, 1], [0, 1], linestyle="--", label="Chance")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title(f"ROC — Mole Present vs Absent ({POOLING}) [RADDINO-s8]")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

prec, rec, _ = precision_recall_curve(y_te, proba)
plt.figure(figsize=(5, 4))
plt.plot(rec, prec, label=f"AUPRC = {auprc:.3f}")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title(f"Precision–Recall — Mole Present vs Absent ({POOLING}) [RADDINO-s8]")
plt.legend(loc="lower left")
plt.tight_layout()
plt.show()

total_time = t_pool_tr + t_pool_te + t_fit + t_pred
print(f"\nTotal wall time ≈ {total_time:.2f}s")


### Random Line 2

In [ ]:
TR_EMB = "/home/jupyter-yin10/Image_Analysis/random_l2/train_emb_rad_dino.npy"
TR_LAB = "/home/jupyter-yin10/Image_Analysis/random_l2/train_labels_rad_dino.npy"
TE_EMB = "/home/jupyter-yin10/Image_Analysis/random_l2/test_emb_rad_dino.npy"
TE_LAB = "/home/jupyter-yin10/Image_Analysis/random_l2/test_labels_rad_dino.npy"

def load_memmap(path):
    return np.load(path, mmap_mode="r", allow_pickle=False)

Xtr_mm = load_memmap(TR_EMB)
Xte_mm = load_memmap(TE_EMB)
y_tr   = np.load(TR_LAB, allow_pickle=False)
y_te   = np.load(TE_LAB, allow_pickle=False)

# ---------- run + time ----------
POOLING = "mean"     # try: "mean_std" for stronger baseline (bigger features)
BATCH   = None       # None => auto (~512MB/batch). Use 64/32 if memory is tight.

X_tr, t_pool_tr = pool_memmap_streaming_timed(Xtr_mm, pooling=POOLING, batch=BATCH, out_dtype=np.float32, desc="Pool train")
X_te, t_pool_te = pool_memmap_streaming_timed(Xte_mm, pooling=POOLING, batch=BATCH, out_dtype=np.float32, desc="Pool test")
print(f"Pooled shapes: train {X_tr.shape}, test {X_te.shape}")

# ---- train & time LogisticRegression (CPU) ----
t0 = time.perf_counter()
probe = make_pipeline(
    StandardScaler(with_mean=True, with_std=True),
    LogisticRegression(
        penalty="l2",
        C=1.0,
        solver="lbfgs",         # try 'saga' if want L1/elasticnet or faster on very large N
        max_iter=2000,
        class_weight="balanced",
        random_state=0,
        n_jobs=-1
    )
)
probe.fit(X_tr, y_tr)
t_fit = time.perf_counter() - t0
print(f"Training done in {t_fit:.2f}s")

# ---- eval & time ----
t0 = time.perf_counter()
proba = probe.predict_proba(X_te)[:, 1]
pred  = (proba >= 0.5).astype(int)
t_pred = time.perf_counter() - t0

acc   = accuracy_score(y_te, pred)
auroc = roc_auc_score(y_te, proba)
auprc = average_precision_score(y_te, proba)
cm    = confusion_matrix(y_te, pred)
print(f"Eval done in {t_pred:.2f}s")
print(f"Accuracy: {acc:.4f} | AUROC: {auroc:.4f} | AUPRC: {auprc:.4f}\nConfusion matrix:\n{cm}")
print(classification_report(y_te, pred, digits=3))

# ---- plots (unchanged) ----
fpr, tpr, _ = roc_curve(y_te, proba)
plt.figure(figsize=(5, 4))
plt.plot(fpr, tpr, label=f"AUC = {auroc:.3f}")
plt.plot([0, 1], [0, 1], linestyle="--", label="Chance")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title(f"ROC — Mole Present vs Absent ({POOLING}) [RADDINO-l2]")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

prec, rec, _ = precision_recall_curve(y_te, proba)
plt.figure(figsize=(5, 4))
plt.plot(rec, prec, label=f"AUPRC = {auprc:.3f}")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title(f"Precision–Recall — Mole Present vs Absent ({POOLING}) [RADDINO-l2]")
plt.legend(loc="lower left")
plt.tight_layout()
plt.show()

total_time = t_pool_tr + t_pool_te + t_fit + t_pred
print(f"\nTotal wall time ≈ {total_time:.2f}s")


### Random Line 4

In [ ]:
TR_EMB = "/home/jupyter-yin10/Image_Analysis/random_l4/train_emb_rad_dino.npy"
TR_LAB = "/home/jupyter-yin10/Image_Analysis/random_l4/train_labels_rad_dino.npy"
TE_EMB = "/home/jupyter-yin10/Image_Analysis/random_l4/test_emb_rad_dino.npy"
TE_LAB = "/home/jupyter-yin10/Image_Analysis/random_l4/test_labels_rad_dino.npy"

def load_memmap(path):
    return np.load(path, mmap_mode="r", allow_pickle=False)

Xtr_mm = load_memmap(TR_EMB)
Xte_mm = load_memmap(TE_EMB)
y_tr   = np.load(TR_LAB, allow_pickle=False)
y_te   = np.load(TE_LAB, allow_pickle=False)

# ---------- run + time ----------
POOLING = "mean"     # try: "mean_std" for stronger baseline (bigger features)
BATCH   = None       # None => auto (~512MB/batch). Use 64/32 if memory is tight.

X_tr, t_pool_tr = pool_memmap_streaming_timed(Xtr_mm, pooling=POOLING, batch=BATCH, out_dtype=np.float32, desc="Pool train")
X_te, t_pool_te = pool_memmap_streaming_timed(Xte_mm, pooling=POOLING, batch=BATCH, out_dtype=np.float32, desc="Pool test")
print(f"Pooled shapes: train {X_tr.shape}, test {X_te.shape}")

# ---- train & time LogisticRegression (CPU) ----
t0 = time.perf_counter()
probe = make_pipeline(
    StandardScaler(with_mean=True, with_std=True),
    LogisticRegression(
        penalty="l2",
        C=1.0,
        solver="lbfgs",         # try 'saga' if want L1/elasticnet or faster on very large N
        max_iter=2000,
        class_weight="balanced",
        random_state=0,
        n_jobs=-1
    )
)
probe.fit(X_tr, y_tr)
t_fit = time.perf_counter() - t0
print(f"Training done in {t_fit:.2f}s")

# ---- eval & time ----
t0 = time.perf_counter()
proba = probe.predict_proba(X_te)[:, 1]
pred  = (proba >= 0.5).astype(int)
t_pred = time.perf_counter() - t0

acc   = accuracy_score(y_te, pred)
auroc = roc_auc_score(y_te, proba)
auprc = average_precision_score(y_te, proba)
cm    = confusion_matrix(y_te, pred)
print(f"Eval done in {t_pred:.2f}s")
print(f"Accuracy: {acc:.4f} | AUROC: {auroc:.4f} | AUPRC: {auprc:.4f}\nConfusion matrix:\n{cm}")
print(classification_report(y_te, pred, digits=3))

# ---- plots (unchanged) ----
fpr, tpr, _ = roc_curve(y_te, proba)
plt.figure(figsize=(5, 4))
plt.plot(fpr, tpr, label=f"AUC = {auroc:.3f}")
plt.plot([0, 1], [0, 1], linestyle="--", label="Chance")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title(f"ROC — Mole Present vs Absent ({POOLING}) [RADDINO-l4]")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

prec, rec, _ = precision_recall_curve(y_te, proba)
plt.figure(figsize=(5, 4))
plt.plot(rec, prec, label=f"AUPRC = {auprc:.3f}")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title(f"Precision–Recall — Mole Present vs Absent ({POOLING}) [RADDINO-l4]")
plt.legend(loc="lower left")
plt.tight_layout()
plt.show()

total_time = t_pool_tr + t_pool_te + t_fit + t_pred
print(f"\nTotal wall time ≈ {total_time:.2f}s")


### Random Line 8

In [ ]:
TR_EMB = "/home/jupyter-yin10/Image_Analysis/random_l8/train_emb_rad_dino.npy"
TR_LAB = "/home/jupyter-yin10/Image_Analysis/random_l8/train_labels_rad_dino.npy"
TE_EMB = "/home/jupyter-yin10/Image_Analysis/random_l8/test_emb_rad_dino.npy"
TE_LAB = "/home/jupyter-yin10/Image_Analysis/random_l8/test_labels_rad_dino.npy"

def load_memmap(path):
    return np.load(path, mmap_mode="r", allow_pickle=False)

Xtr_mm = load_memmap(TR_EMB)
Xte_mm = load_memmap(TE_EMB)
y_tr   = np.load(TR_LAB, allow_pickle=False)
y_te   = np.load(TE_LAB, allow_pickle=False)

# ---------- run + time ----------
POOLING = "mean"     # try: "mean_std" for stronger baseline (bigger features)
BATCH   = None       # None => auto (~512MB/batch). Use 64/32 if memory is tight.

X_tr, t_pool_tr = pool_memmap_streaming_timed(Xtr_mm, pooling=POOLING, batch=BATCH, out_dtype=np.float32, desc="Pool train")
X_te, t_pool_te = pool_memmap_streaming_timed(Xte_mm, pooling=POOLING, batch=BATCH, out_dtype=np.float32, desc="Pool test")
print(f"Pooled shapes: train {X_tr.shape}, test {X_te.shape}")

# ---- train & time LogisticRegression (CPU) ----
t0 = time.perf_counter()
probe = make_pipeline(
    StandardScaler(with_mean=True, with_std=True),
    LogisticRegression(
        penalty="l2",
        C=1.0,
        solver="lbfgs",         # try 'saga' if want L1/elasticnet or faster on very large N
        max_iter=2000,
        class_weight="balanced",
        random_state=0,
        n_jobs=-1
    )
)
probe.fit(X_tr, y_tr)
t_fit = time.perf_counter() - t0
print(f"Training done in {t_fit:.2f}s")

# ---- eval & time ----
t0 = time.perf_counter()
proba = probe.predict_proba(X_te)[:, 1]
pred  = (proba >= 0.5).astype(int)
t_pred = time.perf_counter() - t0

acc   = accuracy_score(y_te, pred)
auroc = roc_auc_score(y_te, proba)
auprc = average_precision_score(y_te, proba)
cm    = confusion_matrix(y_te, pred)
print(f"Eval done in {t_pred:.2f}s")
print(f"Accuracy: {acc:.4f} | AUROC: {auroc:.4f} | AUPRC: {auprc:.4f}\nConfusion matrix:\n{cm}")
print(classification_report(y_te, pred, digits=3))

# ---- plots (unchanged) ----
fpr, tpr, _ = roc_curve(y_te, proba)
plt.figure(figsize=(5, 4))
plt.plot(fpr, tpr, label=f"AUC = {auroc:.3f}")
plt.plot([0, 1], [0, 1], linestyle="--", label="Chance")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title(f"ROC — Mole Present vs Absent ({POOLING}) [RADDINO-l8]")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

prec, rec, _ = precision_recall_curve(y_te, proba)
plt.figure(figsize=(5, 4))
plt.plot(rec, prec, label=f"AUPRC = {auprc:.3f}")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title(f"Precision–Recall — Mole Present vs Absent ({POOLING}) [RADDINO-l8]")
plt.legend(loc="lower left")
plt.tight_layout()
plt.show()

total_time = t_pool_tr + t_pool_te + t_fit + t_pred
print(f"\nTotal wall time ≈ {total_time:.2f}s")
